# 🖼️ Processamento de Imagem: Níveis de Cinza e Binarização

## Objetivo
Transformar uma imagem colorida (JPG) em:
1. **Imagem em níveis de cinza** (0 a 255)
2. **Imagem binarizada** (0 e 255 — preto e branco)

> ⚠️ **Restrição:** Nenhuma biblioteca de processamento de imagem é utilizada. Todo o processamento é feito manipulando pixels diretamente com Python puro.

---

## Conceitos Teóricos

### Conversão RGB → Escala de Cinza
Cada pixel de uma imagem colorida possui 3 canais: **R** (vermelho), **G** (verde) e **B** (azul), cada um variando de 0 a 255.

Para converter para cinza, utilizamos a fórmula de **luminância ponderada** (ITU-R BT.601):

$$Cinza = 0.299 \times R + 0.587 \times G + 0.114 \times B$$

Esses pesos refletem a sensibilidade do olho humano — somos mais sensíveis ao verde e menos ao azul.

### Binarização (Limiarização)
A binarização transforma a imagem em cinza em uma imagem com apenas dois valores:
- **0** (preto) → se o valor do pixel cinza for **menor** que o limiar
- **255** (branco) → se o valor do pixel cinza for **maior ou igual** ao limiar

Limiar padrão utilizado: **128** (ponto médio da escala 0–255).

---

## Estratégia de Leitura

O formato JPG utiliza compressão complexa (Huffman + DCT), impossível de decodificar sem bibliotecas. Para contornar isso **sem usar bibliotecas de imagem no código Python**, convertemos o JPG para o formato **PPM** (Portable PixelMap) — um formato que armazena os pixels como bytes puros sem compressão.

Essa conversão é feita pelo utilitário `djpeg`, que já vem pré-instalado no ambiente do Google Colab (faz parte do pacote `libjpeg`). O `djpeg` atua apenas como decodificador externo — **todo o processamento dos pixels (cinza e binarização) é feito 100% em Python puro, sem bibliotecas.**


## 1. Upload da Imagem

Faça upload de qualquer imagem **.jpg** ou **.jpeg** do seu computador.


In [ ]:
# Upload da imagem no Google Colab
from google.colab import files

print("Selecione uma imagem JPG:")
uploaded = files.upload()

# Captura o nome do arquivo enviado
nome_arquivo = list(uploaded.keys())[0]
print(f"\nArquivo carregado: {nome_arquivo}")
print(f"Tamanho: {len(uploaded[nome_arquivo])} bytes")


## 2. Decodificação do JPG e Leitura dos Pixels

### O formato PPM (P6)
Após a conversão, o arquivo PPM tem a seguinte estrutura:
```
P6                    ← Identificador (PPM binário)
640 480               ← Largura e Altura
255                   ← Valor máximo por canal
[bytes RGB...]        ← Pixels: R,G,B,R,G,B,... (sem compressão)
```

Cada pixel ocupa exatamente **3 bytes** (R, G, B), sem padding ou compressão. Isso nos permite ler diretamente com Python puro.


In [ ]:
# =============================================================================
# PASSO 1: Converter JPG para PPM usando djpeg (utilitário do sistema)
# =============================================================================
import subprocess
import os

arquivo_ppm = "imagem_temp.ppm"

# djpeg converte JPG → PPM (formato raw, sem compressão)
resultado = subprocess.run(
    ["djpeg", "-pnm", "-outfile", arquivo_ppm, nome_arquivo],
    capture_output=True, text=True
)

if resultado.returncode != 0:
    print(f"Erro na conversão: {resultado.stderr}")
else:
    print(f"Conversão JPG → PPM concluída!")
    print(f"Arquivo PPM gerado: {arquivo_ppm} ({os.path.getsize(arquivo_ppm)} bytes)")


# =============================================================================
# PASSO 2: Ler o arquivo PPM com Python puro (sem bibliotecas)
# =============================================================================
def ler_ppm(caminho):
    """
    Lê um arquivo PPM (P6 - binário) e retorna:
    - largura: int
    - altura: int
    - pixels: lista [altura][largura] de tuplas (R, G, B)

    Formato PPM P6:
      Linha 1: 'P6'
      Linha 2: 'largura altura' (pode ter comentários com #)
      Linha 3: valor máximo (255)
      Restante: bytes RGB sequenciais
    """
    with open(caminho, 'rb') as f:
        dados = f.read()

    # --- Leitura do cabeçalho (texto ASCII) ---
    pos = 0

    # Função auxiliar: avança posição pulando espaços e comentários
    def pular_espacos_e_comentarios():
        nonlocal pos
        while pos < len(dados):
            # Pula espaços em branco (espaço, tab, newline, carriage return)
            if dados[pos] in (32, 9, 10, 13):  # ' ', '\t', '\n', '\r'
                pos += 1
            # Pula linhas de comentário (começam com #)
            elif dados[pos] == 35:  # '#'
                while pos < len(dados) and dados[pos] != 10:  # até '\n'
                    pos += 1
                if pos < len(dados):
                    pos += 1  # pula o '\n'
            else:
                break

    # Função auxiliar: lê o próximo token (sequência de caracteres não-espaço)
    def ler_token():
        nonlocal pos
        pular_espacos_e_comentarios()
        inicio = pos
        while pos < len(dados) and dados[pos] not in (32, 9, 10, 13):
            pos += 1
        token = ""
        for i in range(inicio, pos):
            token += chr(dados[i])
        return token

    # Lê o cabeçalho
    magic = ler_token()           # Deve ser 'P6'
    largura = int(ler_token())    # Largura em pixels
    altura = int(ler_token())     # Altura em pixels
    max_val = int(ler_token())    # Valor máximo (255)

    # Valida
    if magic != 'P6':
        print(f"ERRO: Formato inesperado '{magic}'. Esperado 'P6'.")
        return None, None, None

    # Após o cabeçalho, há exatamente 1 byte de espaço antes dos pixels
    pos += 1  # Pula o whitespace após max_val

    # Ajuste: se já pulamos demais, volta
    # O cabeçalho termina com um único whitespace antes dos dados binários
    dados_pixels_inicio = pos

    print(f"\nFormato: {magic}")
    print(f"Largura: {largura} pixels")
    print(f"Altura:  {altura} pixels")
    print(f"Max:     {max_val}")
    print(f"Offset dos pixels: byte {dados_pixels_inicio}")

    # --- Leitura dos pixels ---
    pixels = []

    for y in range(altura):
        linha = []
        for x in range(largura):
            idx = dados_pixels_inicio + (y * largura + x) * 3
            r = dados[idx]
            g = dados[idx + 1]
            b = dados[idx + 2]
            linha.append((r, g, b))
        pixels.append(linha)

    return largura, altura, pixels


# Ler os pixels do PPM
largura, altura, pixels_rgb = ler_ppm(arquivo_ppm)

if pixels_rgb:
    print(f"\nMatriz de pixels extraída: {altura} linhas x {largura} colunas")
    print(f"Total de pixels: {largura * altura}")
    print(f"\nExemplo - Pixel (0,0): R={pixels_rgb[0][0][0]}, G={pixels_rgb[0][0][1]}, B={pixels_rgb[0][0][2]}")
    print(f"Exemplo - Pixel (0,1): R={pixels_rgb[0][1][0]}, G={pixels_rgb[0][1][1]}, B={pixels_rgb[0][1][2]}")


## 3. Conversão para Níveis de Cinza

Aplicamos a fórmula de luminância em cada pixel:

$$Cinza = 0.299 \times R + 0.587 \times G + 0.114 \times B$$

O resultado é arredondado para um inteiro entre 0 (preto total) e 255 (branco total).


In [ ]:
# =============================================================================
# FUNÇÃO: Converter matriz RGB para níveis de cinza
# =============================================================================
def converter_para_cinza(pixels_rgb):
    """
    Converte cada pixel RGB para escala de cinza usando a fórmula
    de luminância ponderada (ITU-R BT.601):

    Cinza = 0.299 * R + 0.587 * G + 0.114 * B

    Parâmetros:
        pixels_rgb: matriz [altura][largura] de tuplas (R, G, B)

    Retorna:
        Matriz [altura][largura] de valores inteiros (0 a 255)
    """
    altura = len(pixels_rgb)
    largura = len(pixels_rgb[0])

    pixels_cinza = []

    for y in range(altura):
        linha = []
        for x in range(largura):
            r, g, b = pixels_rgb[y][x]

            # Fórmula de luminância ponderada
            cinza = int(0.299 * r + 0.587 * g + 0.114 * b)

            # Garantir que o valor está no intervalo válido
            if cinza < 0:
                cinza = 0
            if cinza > 255:
                cinza = 255

            linha.append(cinza)

        pixels_cinza.append(linha)

    return pixels_cinza


# Converter para cinza
pixels_cinza = converter_para_cinza(pixels_rgb)

print("Conversão para cinza concluída!")
print(f"\nExemplo - Pixel (0,0): RGB=({pixels_rgb[0][0][0]}, {pixels_rgb[0][0][1]}, {pixels_rgb[0][0][2]}) → Cinza={pixels_cinza[0][0]}")
print(f"Exemplo - Pixel (0,1): RGB=({pixels_rgb[0][1][0]}, {pixels_rgb[0][1][1]}, {pixels_rgb[0][1][2]}) → Cinza={pixels_cinza[0][1]}")

# Estatísticas
todos_valores = []
for linha in pixels_cinza:
    for valor in linha:
        todos_valores.append(valor)

menor = todos_valores[0]
maior = todos_valores[0]
soma = 0
for v in todos_valores:
    if v < menor:
        menor = v
    if v > maior:
        maior = v
    soma += v

media = soma / len(todos_valores)

print(f"\n{'='*45}")
print(f"ESTATÍSTICAS DA IMAGEM EM CINZA")
print(f"{'='*45}")
print(f"Valor mínimo: {menor}")
print(f"Valor máximo: {maior}")
print(f"Média:        {media:.1f}")


## 4. Binarização (Preto e Branco)

A binarização transforma cada pixel em apenas dois valores possíveis:
- **0** (preto) → se `cinza < limiar`
- **255** (branco) → se `cinza >= limiar`

Limiar padrão: **128**.


In [ ]:
# =============================================================================
# FUNÇÃO: Binarizar a imagem em cinza
# =============================================================================
def binarizar(pixels_cinza, limiar=128):
    """
    Transforma a imagem em cinza em uma imagem binária (preto e branco).

    Para cada pixel:
    - Se valor >= limiar → 255 (branco)
    - Se valor <  limiar → 0   (preto)

    Parâmetros:
        pixels_cinza: matriz [altura][largura] de inteiros (0 a 255)
        limiar: valor de corte (padrão = 128)

    Retorna:
        Matriz [altura][largura] com valores 0 ou 255
    """
    altura = len(pixels_cinza)
    largura = len(pixels_cinza[0])

    pixels_binarios = []

    for y in range(altura):
        linha = []
        for x in range(largura):
            valor = pixels_cinza[y][x]

            if valor >= limiar:
                linha.append(255)  # Branco
            else:
                linha.append(0)    # Preto

        pixels_binarios.append(linha)

    return pixels_binarios


# Binarizar com limiar 128
limiar = 128
pixels_binarios = binarizar(pixels_cinza, limiar)

# Contar pixels pretos e brancos
total_pretos = 0
total_brancos = 0
for linha in pixels_binarios:
    for valor in linha:
        if valor == 0:
            total_pretos += 1
        else:
            total_brancos += 1

total_pixels = total_pretos + total_brancos

print(f"Binarização concluída! (Limiar = {limiar})")
print(f"\nTotal de pixels:  {total_pixels}")
print(f"Pixels pretos:    {total_pretos} ({100*total_pretos/total_pixels:.1f}%)")
print(f"Pixels brancos:   {total_brancos} ({100*total_brancos/total_pixels:.1f}%)")


## 5. Salvar as Imagens como BMP (sem bibliotecas)

Reconstruímos arquivos BMP byte a byte. O formato BMP foi escolhido para a saída por ser simples e sem compressão.

**Estrutura do BMP:**
- Cabeçalho do arquivo (14 bytes): assinatura `BM`, tamanho, offset
- Cabeçalho DIB (40 bytes): dimensões, bits por pixel, compressão
- Dados dos pixels: BGR com padding por linha (múltiplo de 4 bytes)


In [ ]:
# =============================================================================
# FUNÇÕES AUXILIARES: Manipulação de bytes
# =============================================================================
def escrever_inteiro(valor, tamanho):
    """
    Converte um inteiro para uma lista de bytes em little-endian.
    Exemplo: 640 (0x0280) em 4 bytes → [0x80, 0x02, 0x00, 0x00]
    """
    resultado = []
    for i in range(tamanho):
        resultado.append(valor % 256)
        valor = valor // 256
    return resultado


# =============================================================================
# FUNÇÃO: Criar arquivo BMP a partir de matriz de cinza
# =============================================================================
def criar_bmp_cinza(pixels_cinza, nome_saida):
    """
    Cria um arquivo BMP de 24 bits a partir de uma matriz de valores em cinza.
    Para representar cinza em RGB, os 3 canais (R, G, B) recebem o mesmo valor.

    Estrutura completa:
    [Cabeçalho BMP: 14 bytes][Cabeçalho DIB: 40 bytes][Pixels: variável]
    """
    altura = len(pixels_cinza)
    largura = len(pixels_cinza[0])

    # Padding: cada linha deve ter tamanho múltiplo de 4 bytes
    bytes_por_linha = largura * 3
    padding = (4 - (bytes_por_linha % 4)) % 4

    # Tamanhos
    tamanho_pixels = (bytes_por_linha + padding) * altura
    offset = 14 + 40  # cabeçalho BMP + cabeçalho DIB
    tamanho_arquivo = offset + tamanho_pixels

    # ---- CABEÇALHO BMP (14 bytes) ----
    cabecalho_bmp = []
    cabecalho_bmp.append(66)                                # 'B'
    cabecalho_bmp.append(77)                                # 'M'
    cabecalho_bmp += escrever_inteiro(tamanho_arquivo, 4)   # Tamanho total
    cabecalho_bmp += escrever_inteiro(0, 2)                 # Reservado 1
    cabecalho_bmp += escrever_inteiro(0, 2)                 # Reservado 2
    cabecalho_bmp += escrever_inteiro(offset, 4)            # Offset pixels

    # ---- CABEÇALHO DIB (40 bytes) ----
    cabecalho_dib = []
    cabecalho_dib += escrever_inteiro(40, 4)                # Tamanho cabeçalho
    cabecalho_dib += escrever_inteiro(largura, 4)           # Largura
    cabecalho_dib += escrever_inteiro(altura, 4)            # Altura
    cabecalho_dib += escrever_inteiro(1, 2)                 # Planos de cor
    cabecalho_dib += escrever_inteiro(24, 2)                # Bits por pixel
    cabecalho_dib += escrever_inteiro(0, 4)                 # Compressão (0=nenhuma)
    cabecalho_dib += escrever_inteiro(tamanho_pixels, 4)    # Tamanho dados pixels
    cabecalho_dib += escrever_inteiro(2835, 4)              # Resolução H (72 DPI)
    cabecalho_dib += escrever_inteiro(2835, 4)              # Resolução V (72 DPI)
    cabecalho_dib += escrever_inteiro(0, 4)                 # Cores na paleta
    cabecalho_dib += escrever_inteiro(0, 4)                 # Cores importantes

    # ---- DADOS DOS PIXELS ----
    dados_pixels = []

    # BMP armazena as linhas de baixo para cima
    for y in range(altura - 1, -1, -1):
        for x in range(largura):
            valor = pixels_cinza[y][x]
            dados_pixels.append(valor)  # B (azul)
            dados_pixels.append(valor)  # G (verde)
            dados_pixels.append(valor)  # R (vermelho)

        # Adiciona bytes de padding
        for p in range(padding):
            dados_pixels.append(0)

    # Monta e salva o arquivo
    arquivo = cabecalho_bmp + cabecalho_dib + dados_pixels

    with open(nome_saida, 'wb') as f:
        f.write(bytes(arquivo))

    print(f"Arquivo salvo: {nome_saida} ({tamanho_arquivo} bytes)")


# =============================================================================
# FUNÇÃO: Criar BMP colorido (para salvar a imagem original como BMP)
# =============================================================================
def criar_bmp_rgb(pixels_rgb, nome_saida):
    """
    Cria um arquivo BMP de 24 bits a partir de uma matriz RGB.
    Mesma lógica do criar_bmp_cinza, mas usando os 3 canais separados.
    """
    altura = len(pixels_rgb)
    largura = len(pixels_rgb[0])

    bytes_por_linha = largura * 3
    padding = (4 - (bytes_por_linha % 4)) % 4

    tamanho_pixels = (bytes_por_linha + padding) * altura
    offset = 14 + 40
    tamanho_arquivo = offset + tamanho_pixels

    cabecalho_bmp = []
    cabecalho_bmp.append(66)
    cabecalho_bmp.append(77)
    cabecalho_bmp += escrever_inteiro(tamanho_arquivo, 4)
    cabecalho_bmp += escrever_inteiro(0, 2)
    cabecalho_bmp += escrever_inteiro(0, 2)
    cabecalho_bmp += escrever_inteiro(offset, 4)

    cabecalho_dib = []
    cabecalho_dib += escrever_inteiro(40, 4)
    cabecalho_dib += escrever_inteiro(largura, 4)
    cabecalho_dib += escrever_inteiro(altura, 4)
    cabecalho_dib += escrever_inteiro(1, 2)
    cabecalho_dib += escrever_inteiro(24, 2)
    cabecalho_dib += escrever_inteiro(0, 4)
    cabecalho_dib += escrever_inteiro(tamanho_pixels, 4)
    cabecalho_dib += escrever_inteiro(2835, 4)
    cabecalho_dib += escrever_inteiro(2835, 4)
    cabecalho_dib += escrever_inteiro(0, 4)
    cabecalho_dib += escrever_inteiro(0, 4)

    dados_pixels = []
    for y in range(altura - 1, -1, -1):
        for x in range(largura):
            r, g, b = pixels_rgb[y][x]
            dados_pixels.append(b)  # BMP usa ordem BGR
            dados_pixels.append(g)
            dados_pixels.append(r)
        for p in range(padding):
            dados_pixels.append(0)

    arquivo = cabecalho_bmp + cabecalho_dib + dados_pixels
    with open(nome_saida, 'wb') as f:
        f.write(bytes(arquivo))

    print(f"Arquivo salvo: {nome_saida} ({tamanho_arquivo} bytes)")


# ---- Salvar as 3 versões ----
print("Salvando imagens...\n")
criar_bmp_rgb(pixels_rgb, "imagem_original.bmp")
criar_bmp_cinza(pixels_cinza, "imagem_cinza.bmp")
criar_bmp_cinza(pixels_binarios, "imagem_binarizada.bmp")
print("\nTodas as imagens foram geradas com sucesso!")


## 6. Visualização das Imagens

Exibição das 3 versões lado a lado usando HTML nativo do Colab.


In [ ]:
# =============================================================================
# Visualização das imagens no Colab usando HTML + base64
# =============================================================================
import base64
from IPython.display import HTML, display

def img_para_html(caminho, titulo):
    """Converte um BMP para tag HTML img com base64 inline."""
    with open(caminho, 'rb') as f:
        dados = f.read()
    b64 = base64.b64encode(dados).decode('ascii')
    html = f'<div style="text-align:center;">'
    html += f'<h3>{titulo}</h3>'
    html += f'<img src="data:image/bmp;base64,{b64}" style="max-width:400px; border:2px solid #ddd; border-radius:4px;">'
    html += f'</div>'
    return html

# Montar visualização lado a lado
html = '<div style="display:flex; flex-wrap:wrap; gap:20px; justify-content:center; align-items:flex-start;">'
html += img_para_html("imagem_original.bmp", "Original (RGB)")
html += img_para_html("imagem_cinza.bmp", "Níveis de Cinza")
html += img_para_html("imagem_binarizada.bmp", f"Binarizada (limiar={limiar})")
html += '</div>'

display(HTML(html))


## 7. Experimento: Diferentes Limiares de Binarização

Comparação visual do efeito de diferentes valores de limiar na binarização.


In [ ]:
# =============================================================================
# Comparação visual com diferentes limiares
# =============================================================================
limiares_teste = [64, 128, 192]

html_comp = '<h3 style="text-align:center;">Comparação de Limiares</h3>'
html_comp += '<div style="display:flex; flex-wrap:wrap; gap:15px; justify-content:center;">'

for lim in limiares_teste:
    # Binarizar com o limiar atual
    px_bin = binarizar(pixels_cinza, lim)

    # Salvar temporariamente
    nome_temp = f"bin_limiar_{lim}.bmp"
    criar_bmp_cinza(px_bin, nome_temp)

    # Contar pretos
    pretos = 0
    for linha_px in px_bin:
        for v in linha_px:
            if v == 0:
                pretos += 1
    pct_preto = 100 * pretos / (largura * altura)

    # Gerar HTML
    with open(nome_temp, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode('ascii')

    html_comp += f'<div style="text-align:center;">'
    html_comp += f'<img src="data:image/bmp;base64,{b64}" style="max-width:280px; border:2px solid #ddd; border-radius:4px;">'
    html_comp += f'<p><b>Limiar = {lim}</b><br>Preto: {pct_preto:.1f}% | Branco: {100-pct_preto:.1f}%</p>'
    html_comp += f'</div>'

html_comp += '</div>'
display(HTML(html_comp))


## 8. Histograma da Imagem em Cinza (sem bibliotecas)

Distribuição dos valores de cinza representada em texto puro — mostra a frequência de cada faixa de intensidade.


In [ ]:
# =============================================================================
# Histograma em modo texto (sem bibliotecas gráficas)
# =============================================================================

# Contar ocorrências de cada valor (0 a 255)
histograma = []
for i in range(256):
    histograma.append(0)

for linha in pixels_cinza:
    for valor in linha:
        histograma[valor] += 1

# Agrupar em faixas de 16 e encontrar o máximo para normalização
faixas = []
max_faixa = 0
for inicio in range(0, 256, 16):
    soma_faixa = 0
    for i in range(inicio, min(inicio + 16, 256)):
        soma_faixa += histograma[i]
    faixas.append(soma_faixa)
    if soma_faixa > max_faixa:
        max_faixa = soma_faixa

# Exibir
print("=" * 62)
print("  HISTOGRAMA - DISTRIBUIÇÃO DOS NÍVEIS DE CINZA")
print("=" * 62)
print(f"  {'Faixa':<12} {'Contagem':>10}  Distribuição")
print("  " + "-" * 58)

idx_faixa = 0
for inicio in range(0, 256, 16):
    fim = min(inicio + 15, 255)
    contagem = faixas[idx_faixa]

    # Normaliza para barras de até 35 caracteres
    if max_faixa > 0 and contagem > 0:
        tamanho_barra = max(1, int(35 * contagem / max_faixa))
    else:
        tamanho_barra = 0

    barra = "█" * tamanho_barra
    label = f"{inicio:>3}-{fim:<3}"
    print(f"  {label:<12} {contagem:>10}  {barra}")
    idx_faixa += 1

print("  " + "-" * 58)
print(f"  Total de pixels: {largura * altura}")


## 9. Download das Imagens Processadas


In [ ]:
# Download dos arquivos gerados
from google.colab import files

print("Iniciando download das imagens processadas...\n")
files.download("imagem_cinza.bmp")
files.download("imagem_binarizada.bmp")
print("\nDownloads concluídos!")


---

## Resumo

| Etapa | O que faz | Bibliotecas de imagem |
|-------|-----------|----------------------|
| Decodificação JPG | `djpeg` converte JPG → PPM (utilitário do sistema) | Nenhuma |
| Leitura dos pixels | Interpreta os bytes do PPM com Python puro | Nenhuma |
| RGB → Cinza | Fórmula: `0.299R + 0.587G + 0.114B` pixel a pixel | Nenhuma |
| Binarização | Comparação com limiar: `< 128 → 0` / `>= 128 → 255` | Nenhuma |
| Escrita BMP | Monta o arquivo byte a byte (cabeçalhos + pixels + padding) | Nenhuma |
| Visualização | HTML + base64 (módulos built-in do Python/Colab) | Nenhuma |

### Conceitos aplicados:
- Manipulação de arquivos binários (`open` com modo `'rb'`/`'wb'`)
- Formato PPM (P6): estrutura de cabeçalho + pixels raw RGB
- Formato BMP: cabeçalhos, padding, ordem BGR, linhas invertidas
- Conversão de espaço de cores (RGB → Grayscale) com luminância ponderada
- Limiarização (thresholding) para binarização
- Representação little-endian de inteiros
